In [1]:
import math

# Part 1: Knowledge Base
applicant_facts = {
    "profile": {"fico": 635, "dti": 0.43, "income": 4700},
    "status": {"employed": True, "past_defaults": 0},
    "request": {"loan_amount": 10000}
}

# Part 2: Rule-Based Reasoning (RBR)
def rule_fico_reject(facts):
    if facts["profile"]["fico"] < 600:
        return "REJECT: FICO below 600 minimum."
    return None

def rule_dti_review(facts):
    if facts["profile"]["dti"] > 0.40:
        return "FLAG: DTI > 40%. Requires manual underwriting review."
    return None

def rule_income_approve(facts):
    if facts["profile"]["income"] > 8000 and facts["profile"]["dti"] < 0.30:
        return "FAST-TRACK: High income, low DTI."
    return None

def run_inference_engine(facts):
    rules = [rule_fico_reject, rule_dti_review, rule_income_approve]
    inferences = []

    for rule in rules:
        result = rule(facts)
        if result:
            inferences.append(result)

    return inferences if inferences else ["APPROVE: Standard processing."]

# Part 3: Case-Based Reasoning (CBR)
case_base = [
    {"id": 1, "fico": 750, "dti": 0.20, "income": 9000, "defaulted": False, "limit": 25000},
    {"id": 2, "fico": 610, "dti": 0.48, "income": 4000, "defaulted": True,  "limit": 5000},
    {"id": 3, "fico": 680, "dti": 0.35, "income": 6000, "defaulted": False, "limit": 15000},
    {"id": 4, "fico": 590, "dti": 0.50, "income": 3500, "defaulted": True,  "limit": 2000},
    {"id": 5, "fico": 720, "dti": 0.25, "income": 8500, "defaulted": False, "limit": 20000},
    {"id": 6, "fico": 640, "dti": 0.42, "income": 4800, "defaulted": True,  "limit": 8000},
    {"id": 7, "fico": 800, "dti": 0.15, "income": 12000,"defaulted": False, "limit": 50000},
    {"id": 8, "fico": 660, "dti": 0.38, "income": 5200, "defaulted": False, "limit": 10000},
    {"id": 9, "fico": 630, "dti": 0.44, "income": 4600, "defaulted": True,  "limit": 6000},
    {"id": 10,"fico": 690, "dti": 0.31, "income": 7000, "defaulted": False, "limit": 18000}
]

# risk weights
WEIGHTS = {"fico": 1.5, "dti": 2.0, "income": 0.5}

def get_stats(feature):
    # get mean and std dev
    vals = [c[feature] for c in case_base]
    mean = sum(vals) / len(vals)
    variance = sum((x - mean) ** 2 for x in vals) / len(vals)
    return mean, math.sqrt(variance) if variance > 0 else 1

def calc_distance(target, case, stats):
    # calc weighted z-score distance
    dist = 0
    for feat in WEIGHTS:
        mean, std = stats[feat]
        z_target = (target[feat] - mean) / std
        z_case = (case[feat] - mean) / std
        dist += WEIGHTS[feat] * ((z_target - z_case) ** 2)
    return math.sqrt(dist)

def run_cbr(applicant):
    print(f"\n[CBR Engine]")

    # 1. get stats
    stats = {f: get_stats(f) for f in WEIGHTS}

    # 2. retrieve match
    best_match = min(case_base, key=lambda c: calc_distance(applicant["profile"], c, stats))
    min_dist = calc_distance(applicant["profile"], best_match, stats)

    print(f"Retrieve: Match ID {best_match['id']} (Dist: {min_dist:.4f})")
    print(f"Data: FICO {best_match['fico']}, DTI {best_match['dti']}, Income {best_match['income']}")
    print(f"Outcome: {'Defaulted' if best_match['defaulted'] else 'Paid'} | Limit: {best_match['limit']}")

    # 3. reuse & revise limit
    if best_match["defaulted"]:
        adapted_limit = 0
        print("Revise: Match defaulted. Target limit set to 0.")
    else:
        # scale limit
        ratio = applicant["profile"]["income"] / best_match["income"]
        adapted_limit = int(best_match["limit"] * ratio)
        print(f"Revise: Match paid. Target limit adapted to {adapted_limit}")

    # 4. retain case
    new_case = {
        "id": len(case_base) + 1,
        "fico": applicant["profile"]["fico"],
        "dti": applicant["profile"]["dti"],
        "income": applicant["profile"]["income"],
        "defaulted": False,
        "limit": adapted_limit
    }
    case_base.append(new_case)
    print(f"Retain: Saved. New DB size: {len(case_base)}")

if __name__ == "__main__":
    print("[RBR Engine]")
    for res in run_inference_engine(applicant_facts):
        print(f"Rule Fired: {res}")

    run_cbr(applicant_facts)

[RBR Engine]
Rule Fired: FLAG: DTI > 40%. Requires manual underwriting review.

[CBR Engine]
Retrieve: Match ID 6 (Dist: 0.1620)
Data: FICO 640, DTI 0.42, Income 4800
Outcome: Defaulted | Limit: 8000
Revise: Match defaulted. Target limit set to 0.
Retain: Saved. New DB size: 11
